# scipy.optimize / interpolate / signal — small worked examples

Every tool is shown first on a problem you can solve in your head, then on something from
the energy data.

**What's in here**
- `minimize`: find the x that makes f(x) smallest
- `curve_fit`: fit a model function to points, with standard errors
- `minimize_scalar` and root finding with `brentq`
- constrained optimisation: `minimize` with bounds and constraints, and `linprog`
- interpolation: `np.interp`, `interp1d`
- detrending and smoothing
- distances and sparse matrices (brief)

In [1]:
import numpy as np
import pandas as pd
from scipy import optimize, interpolate, signal, stats

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

## 1. `minimize`

f(x) = (x − 3)². The minimum is obviously at x = 3. Watch what `minimize` returns.

In [2]:
def f(x):
    return (x[0] - 3) ** 2          # x arrives as an array, even in 1-D

x0 = np.array([0.0])                # starting guess
res = optimize.minimize(f, x0)
res

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 2.5388963550532293e-16
        x: [ 3.000e+00]
      nit: 2
      jac: [-1.697e-08]
 hess_inv: [[ 5.000e-01]]
     nfev: 6
     njev: 3

In [3]:
print("x       :", res.x)           # where the minimum is
print("fun     :", res.fun)         # f at that point
print("success :", res.success)
print("nit     :", res.nit)         # iterations used
print("message :", res.message)

x       : [3.]
fun     : 2.5388963550532293e-16
success : True
nit     : 2
message : Optimization terminated successfully.


Always read `success` and `message`. A result object is returned even when the optimiser failed.

Two variables: f(x, y) = (x − 1)² + (y + 2)². Minimum at (1, −2).

In [4]:
def g(v):
    return (v[0] - 1) ** 2 + (v[1] + 2) ** 2

res = optimize.minimize(g, x0=[0.0, 0.0])
print("x:", res.x, " fun:", round(res.fun, 8), " success:", res.success)

x: [ 1. -2.]  fun: 0.0  success: True


A custom loss on data: fit a straight line by minimising the sum of squared errors yourself.
Five points near y = 2x + 1.

In [5]:
x = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y = np.array([1.2, 2.8, 5.1, 7.2, 8.9])

def sse(params):
    slope, intercept = params
    pred = slope * x + intercept
    return ((y - pred) ** 2).sum()

print("sse at (0, 0):", sse([0.0, 0.0]))
print("sse at (2, 1):", sse([2.0, 1.0]))

sse at (0, 0): 166.34
sse at (2, 1): 0.13999999999999996


In [6]:
res = optimize.minimize(sse, x0=[0.0, 0.0])
print("slope, intercept:", res.x, " sse:", round(res.fun, 4))
print("np.polyfit      :", np.polyfit(x, y, 1))

slope, intercept: [1.98 1.08]  sse: 0.128
np.polyfit      : [1.98 1.08]


Same answer as least squares. The point of `minimize` is that you can change the loss:
absolute errors instead of squared ones gives a fit that ignores outliers.

In [7]:
y_out = y.copy()
y_out[4] = 30.0                     # one bad point

def sae(params):
    slope, intercept = params
    return np.abs(y_out - (slope * x + intercept)).sum()

print("squared loss fit :", np.polyfit(x, y_out, 1))
print("absolute loss fit:", optimize.minimize(sae, x0=[0.0, 0.0], method="Nelder-Mead").x)

squared loss fit : [ 6.2  -3.14]
absolute loss fit: [2.2977 0.5046]


## 2. `curve_fit`

You give a model function `model(x, param1, param2, ...)` and the data; it returns the
parameters and their covariance. Five points from y = 2x + 1 plus noise.

In [8]:
def line(x, slope, intercept):
    return slope * x + intercept

x = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y = np.array([1.2, 2.8, 5.1, 7.2, 8.9])
popt, pcov = optimize.curve_fit(line, x, y)
print("popt (slope, intercept):", popt)
print("pcov:")
print(pcov)

popt (slope, intercept): [1.98 1.08]
pcov:
[[ 0.0043 -0.0085]
 [-0.0085  0.0256]]


The diagonal of `pcov` holds the variances of the parameters; square roots are the standard errors.

In [9]:
se = np.sqrt(np.diag(pcov))
print("slope     :", round(popt[0], 4), "+/-", round(se[0], 4))
print("intercept :", round(popt[1], 4), "+/-", round(se[1], 4))

slope     : 1.98 +/- 0.0653
intercept : 1.08 +/- 0.16


A nonlinear model: exponential decay y = a · exp(−b x). Starting values (`p0`) matter for
nonlinear fits; a bad start can land in a wrong solution or fail.

In [10]:
def decay(x, a, b):
    return a * np.exp(-b * x)

x = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])
y_true = decay(x, 5.0, 0.7)
y = y_true + np.array([0.1, -0.1, 0.05, -0.05, 0.02, -0.02])
popt, pcov = optimize.curve_fit(decay, x, y, p0=[1.0, 0.1])
print("true a, b  : 5.0 0.7")
print("fitted a, b:", popt)

true a, b  : 5.0 0.7
fitted a, b: [5.0786 0.7218]


On real data: the heating relationship. Daily consumption vs daily temperature, model
`base + slope · max(T_ref − temp, 0)` with three unknowns.

In [11]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
daily = df.resample("D").mean()
temp = daily["temp_c"].values
cons = daily["consumption_mwh"].values

def heating(t, base, slope, t_ref):
    return base + slope * np.clip(t_ref - t, 0, None)

popt, pcov = optimize.curve_fit(heating, temp, cons, p0=[28000, 300, 15])
se = np.sqrt(np.diag(pcov))
for name, val, s in zip(["base", "slope", "t_ref"], popt, se):
    print(f"{name:6s} {val:10.1f}  +/- {s:.1f}")

base      27229.0  +/- 76.0
slope       366.5  +/- 10.3
t_ref        14.9  +/- 0.3


The data were generated with a heating slope of 380 MWh per degree below 15 °C; the fit
recovers roughly that. (Daily means blur the kink slightly.)

**Pitfall:** those standard errors assume independent errors. Consecutive days are
correlated, so the real uncertainty is larger. See `03_scipy/04` for the correction.

## 3. `minimize_scalar` and root finding

One variable, no gradient needed: `minimize_scalar`.

In [12]:
def h(x):
    return (x - 2) ** 2 + 1

res = optimize.minimize_scalar(h)
print("x:", round(res.x, 6), " fun:", round(res.fun, 6))

x: 2.0  fun: 1.0


Root finding: where does f(x) = x² − 4 cross zero? `brentq` needs a bracket [a, b]
where f(a) and f(b) have opposite signs.

In [13]:
def f(x):
    return x ** 2 - 4

a, b = 0.0, 5.0
print("f(a) =", f(a), "  f(b) =", f(b), "  -> opposite signs, bracket is valid")
root = optimize.brentq(f, a, b)
print("root:", root)

f(a) = -4.0   f(b) = 21.0   -> opposite signs, bracket is valid
root: 1.999999999999977


In [14]:
try:
    optimize.brentq(f, 3.0, 5.0)     # both positive: no sign change
except ValueError as e:
    print("ValueError:", e)

ValueError: f(a) and f(b) must have different signs


Practical use: "at what temperature does the heating model predict 32,000 MWh/day?"
Solve heating(t) − 32000 = 0.

In [15]:
def gap(t):
    return heating(t, *popt) - 32000

print("gap(-5) =", round(gap(-5.0)), "  gap(15) =", round(gap(15.0)))
t_star = optimize.brentq(gap, -5.0, 15.0)
print("temperature for 32,000 MWh/day:", round(t_star, 2), "C")

gap(-5) = 2510   gap(15) = -4771
temperature for 32,000 MWh/day: 1.85 C


## 4. Constrained optimisation

Two generators. Gas costs 80 €/MWh, coal 60 €/MWh. Demand is 100 MWh. Gas can produce up
to 70, coal up to 50. Minimise cost.

By hand: use the cheap one (coal) fully: 50 MWh. The rest (50) from gas.
Cost = 50 · 60 + 50 · 80 = 3000 + 4000 = 7000.

In [16]:
cost = np.array([80.0, 60.0])            # [gas, coal] cost per MWh
A_eq = np.array([[1.0, 1.0]])            # gas + coal ...
b_eq = np.array([100.0])                 # ... = 100
bounds = [(0, 70), (0, 50)]              # capacity of each
print("c    :", cost)
print("A_eq :", A_eq)
print("b_eq :", b_eq)

c    : [80. 60.]
A_eq : [[1. 1.]]
b_eq : [100.]


In [17]:
res = optimize.linprog(c=cost, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
print("status  :", res.status, "(0 = optimal)")
print("x       :", res.x, " (gas, coal)")
print("cost    :", res.fun)

status  : 0 (0 = optimal)
x       : [50. 50.]  (gas, coal)
cost    : 7000.0


Exactly the hand answer. `linprog` **minimises**; to maximise, negate the objective.

The dual value of the demand constraint is the cost of one more MWh: gas is the marginal
unit, so it is 80.

In [18]:
print("dual of demand constraint:", res.eqlin.marginals)

dual of demand constraint: [80.]


The same problem with `minimize` (works for nonlinear objectives too, but is slower and
needs a starting point).

In [19]:
def total_cost(x):
    return cost @ x

constraints = [{"type": "eq", "fun": lambda x: x.sum() - 100}]
res = optimize.minimize(total_cost, x0=[50.0, 50.0], bounds=bounds, constraints=constraints)
print("x:", res.x, " cost:", round(res.fun, 2), " success:", res.success)

x: [50. 50.]  cost: 7000.0  success: True


**Pitfall:** `linprog` bounds default to (0, None). Forgetting a capacity bound makes the
solver use the cheap unit for everything.

In [20]:
res = optimize.linprog(c=cost, A_eq=A_eq, b_eq=b_eq, method="highs")   # no bounds
print("x without capacity limits:", res.x)

x without capacity limits: [  0. 100.]


## 5. Interpolation

`np.interp(x_new, x_known, y_known)` draws straight lines between known points.
Three known points, ask for values in between.

In [21]:
x_known = np.array([0.0, 1.0, 2.0])
y_known = np.array([10.0, 20.0, 40.0])
x_new = np.array([0.5, 1.5, 1.75])
np.interp(x_new, x_known, y_known)

array([15., 30., 35.])

0.5 is halfway between 0 and 1 → halfway between 10 and 20 = 15. 1.5 → 30. 1.75 → 35.

**Pitfall:** `np.interp` needs `x_known` sorted ascending. Unsorted input gives garbage without an error.

In [22]:
np.interp(x_new, np.array([2.0, 0.0, 1.0]), np.array([40.0, 10.0, 20.0]))   # unsorted -> wrong

array([40., 20., 20.])

`interp1d` builds a reusable function and supports other kinds (`"cubic"`).

In [23]:
lin = interpolate.interp1d(x_known, y_known, kind="linear")
print("linear at 1.5:", lin(1.5))

linear at 1.5: 30.0


On a Series, `interpolate()` fills NaN gaps by time.

In [24]:
s = pd.Series([10.0, np.nan, np.nan, 40.0], index=pd.date_range("2023-01-01", periods=4, freq="h"))
pd.DataFrame({"s": s, "interpolated": s.interpolate(method="time")})

,s,interpolated
2023-01-01 00:00:00,10.0,10.0
2023-01-01 01:00:00,NaN,20.0
2023-01-01 02:00:00,NaN,30.0
2023-01-01 03:00:00,40.0,40.0


## 6. Detrending and smoothing

A short series with a trend: 1, 2, 3, ... plus a wiggle. `signal.detrend` removes the
straight-line trend.

In [25]:
y = np.array([1.0, 2.5, 2.8, 4.2, 4.9, 6.3])
print("original :", y)
print("detrended:", signal.detrend(y))

original : [1.  2.5 2.8 4.2 4.9 6.3]
detrended: [-0.1095  0.3876 -0.3152  0.0819 -0.221   0.1762]


A moving average smooths. With pandas: `rolling(3, center=True).mean()`.

In [26]:
s = pd.Series([1.0, 5.0, 2.0, 6.0, 3.0, 7.0])
pd.DataFrame({"s": s, "rolling3": s.rolling(3, center=True).mean()})

,s,rolling3
0,1.0,NaN
1,5.0,2.666667
2,2.0,4.333333
3,6.0,3.666667
4,3.0,5.333333
5,7.0,NaN


`center=True` places the average at the middle of the window (fine for plots, **not** for
features: it uses the future).

## 7. Distances and sparse matrices (brief)

`cdist` computes the distance between every pair of rows of two matrices.

In [27]:
from scipy.spatial.distance import cdist
A = np.array([[0.0, 0.0], [1.0, 0.0]])
B = np.array([[0.0, 1.0], [3.0, 4.0]])
cdist(A, B)                           # 2 x 2: distance from each row of A to each row of B

array([[1.    , 5.    ],
       [1.4142, 4.4721]])

Row 0 of A (0,0) to row 1 of B (3,4): √(9 + 16) = 5. Use it to find the days most similar
to today's load profile.

Sparse matrices store only non-zero entries. A one-hot of hour-of-day is 17,520 × 24 with
one non-zero per row.

In [28]:
from scipy import sparse
hours = df.index.hour.values
one_hot = sparse.csr_matrix((np.ones(len(hours)), (np.arange(len(hours)), hours)))
print("shape:", one_hot.shape, " stored values:", one_hot.nnz)

shape: (17520, 24)  stored values: 17520


## Quick reference

| Task | Call |
|---|---|
| minimise f(params) | `optimize.minimize(f, x0)` → check `.success`, `.x`, `.fun` |
| fit a model function | `optimize.curve_fit(model, x, y, p0=...)` → `popt`, `np.sqrt(np.diag(pcov))` |
| 1-D minimum | `optimize.minimize_scalar(f)` |
| root with a bracket | `optimize.brentq(f, a, b)` |
| linear program | `optimize.linprog(c, A_ub, b_ub, A_eq, b_eq, bounds, method="highs")` |
| duals | `res.eqlin.marginals`, `res.ineqlin.marginals` |
| linear interpolation | `np.interp(x_new, x_sorted, y)` |
| fill NaN gaps by time | `s.interpolate(method="time")` |
| remove linear trend | `signal.detrend(y)` |
| pairwise distances | `scipy.spatial.distance.cdist(A, B)` |